In [1]:
from config import init_env
from config import variables
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()


### Langchain Integration
https://help.sap.com/doc/generative-ai-hub-sdk/CLOUD/en-US/_reference/gen_ai_hub.html#langchain-integration

#### Harmonized Model Initialization
The init_llm and init_embedding_model functions allow easy initialization of langchain model interfaces in a harmonized way in generative AI hub sdk

In [2]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from gen_ai_hub.proxy.langchain.init_models import init_llm

template = """Question: {question}
    Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=['question'])
question = 'What is a supernova?'

llm = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=800
)
chain = prompt | llm | StrOutputParser()
response = chain.invoke({'question': question})
print(response)

A supernova is a powerful and luminous explosion that occurs at the end of a star's life cycle. Let's break it down step by step:

1. **Star's Life Cycle**: Stars are massive celestial bodies composed primarily of hydrogen and helium. They generate energy through nuclear fusion, converting hydrogen into helium in their cores. This process releases a tremendous amount of energy, which counteracts the gravitational forces trying to collapse the star.

2. **End of Fusion**: As a star exhausts its nuclear fuel, it can no longer sustain fusion reactions in its core. For massive stars, this typically means they have fused elements up to iron, beyond which fusion is not energetically favorable.

3. **Core Collapse**: Without the outward pressure from fusion, the core of the star begins to collapse under its own gravity. This collapse happens extremely rapidly, often in a matter of seconds.

4. **Explosion**: The core collapse results in a rebound effect, where the outer layers of the star are

init_embedding_model

In [3]:
from gen_ai_hub.proxy.langchain.init_models import init_embedding_model

text = 'Every decoding is another encoding.'

embeddings = init_embedding_model('text-embedding-3-large')
response = embeddings.embed_query(text)
#print(response)


#### Chat model

In [4]:
from langchain_core.prompts.chat import (
    AIMessagePromptTemplate,
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)

from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client

proxy_client = get_proxy_client('gen-ai-hub')

chat_llm = ChatOpenAI(proxy_model_name='gpt-4o', proxy_client=proxy_client)

template = 'You are a helpful assistant that translates english to Chinese.'
system_message_prompt = SystemMessagePromptTemplate.from_template(template)

example_human = HumanMessagePromptTemplate.from_template('Hi')
example_ai = AIMessagePromptTemplate.from_template('Ahoy!')
human_template = '{text}'

human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)
chat_prompt = ChatPromptTemplate.from_messages(
    [system_message_prompt, example_human, example_ai, human_message_prompt])

chain = chat_prompt | chat_llm

response = chain.invoke({'text': 'I love planking.'})
print(response.content)


我喜欢平板支撑。


#### Structured model outputs

In [5]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
from langchain_core.prompts.chat import HumanMessage
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int
chat_model = ChatOpenAI(proxy_model_name="gpt-4o", proxy_client=get_proxy_client())
chat_model = chat_model.with_structured_output(method="json_schema", schema=Person, strict=True)

message = HumanMessage(content="Tell me about a person named John who is 30")
print(chat_model.invoke([message]))


name='John Doe' age=30


### Agent

#### Define tools

In [90]:
from langchain.tools import tool
@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

#### Define agents

In [92]:
from langchain.agents import create_agent
from gen_ai_hub.proxy.langchain.init_models import init_llm
 
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=800
)


agent = create_agent(
    model,
    tools = [search, get_weather],
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

#### Invoke agents

In [93]:

response =agent.invoke(
    {
        "messages":[
            {
                "role": "user",
                "content": "What is weather in Shanghai?"
            }
        ]
    }
)
print(response)

 

{'messages': [HumanMessage(content='What is weather in Shanghai?', additional_kwargs={}, response_metadata={}, id='08208c2f-5637-4770-b75a-56075bc7d3b4'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_enZUAh8J7rtbu8Gmz10z1GKv', 'function': {'arguments': '{"location":"Shanghai"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 78, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-Cney9OvMlY5Za777mVTiZge16h3wu', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b2aea-d8f2-7880-aa7c-ee6fa975c560-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Shanghai'}, 'id': 'c

Define a function to return message directly.

In [94]:

def invoke_agent_messages(agent, content: str):
    payload = {
        "messages": [
            {
                "role": "user", 
                "content": content
            }
        ]
    }
    response = agent.invoke(payload)
    return response["messages"]  # 若缺失会直接抛 KeyError


To check the contect in an easier way, we create a function to find out and then print out key information based on the structure of this message.

In [95]:
import json

def print_message_pairs(messages, verbose=False):
    """
    自动从消息序列中提取：
      - user_query：第一个 HumanMessage 的 content
      - tool_name：第一个 AIMessage.additional_kwargs.tool_calls[0].function.name
      - tool_output：第一个 ToolMessage 的 content
      - assistant_text：最后一个 AIMessage 的 content

    参数：
      - messages: 消息序列（包含 HumanMessage / AIMessage / ToolMessage 等）
      - verbose (bool): 
          True  -> 打印 JSON（包含四个键值）
          False -> 仅打印 assistant_text

    返回：
      - pairs (dict): 以上四个字段的字典，便于后续使用
    """
    # 安全提取工具名
    def extract_tool_name_from_ai(ai_msg):
        ak = getattr(ai_msg, "additional_kwargs", {})
        if isinstance(ak, dict):
            tool_calls = ak.get("tool_calls") or []
            if tool_calls:
                fn = tool_calls[0].get("function") or {}
                return fn.get("name")
        return None

    # 初始化
    user_query = None
    tool_name = None
    tool_output = None
    assistant_text = None

    # 1) 找第一个 HumanMessage 作为用户文本
    for m in messages:
        if m.__class__.__name__ == "HumanMessage":
            user_query = getattr(m, "content", None)
            break

    # 2) 找第一个 AIMessage 中的 tool_calls 取函数名
    for m in messages:
        if m.__class__.__name__ == "AIMessage":
            tool_name = extract_tool_name_from_ai(m)
            if tool_name:
                break

    # 3) 找第一个 ToolMessage 的输出
    for m in messages:
        if m.__class__.__name__ == "ToolMessage":
            tool_output = getattr(m, "content", None)
            break

    # 4) 找最后一个 AIMessage 的最终回复
    for m in reversed(messages):
        if m.__class__.__name__ == "AIMessage":
            assistant_text = getattr(m, "content", None)
            break

    pairs = {
        "user_query": user_query,
        "tool_name": tool_name,
        "tool_output": tool_output,
        "assistant_text": assistant_text,
    }

    # 根据 verbose 控制打印
    if verbose:
        # 打印完整 JSON；ensure_ascii=False 支持中文直出（可按需移除）
        print(json.dumps(pairs, indent=2, ensure_ascii=False))
    else:
        # 仅打印最终助手回复；为防 None，做一下空串兜底
        print(assistant_text or "")




Now it is easier for us to see that, in the below case the tool [search] is not applied.

In [102]:
messages = invoke_agent_messages(agent, "Where is the location of Shanghai?")
print_message_pairs(messages,verbose=True)

{
  "user_query": "Where is the location of Shanghai?",
  "tool_name": null,
  "tool_output": null,
  "assistant_text": "Shanghai is located on the eastern coast of China, at the mouth of the Yangtze River. It is situated in the central part of the country, facing the East China Sea."
}


To let agent to use the tool, change the question closer to the tool description.

In [103]:
messages = invoke_agent_messages(agent, "Search for the location of Shanghai")
print_message_pairs(messages,verbose=True)

{
  "user_query": "Search for the location of Shanghai",
  "tool_name": "search",
  "tool_output": "Results for: location of Shanghai",
  "assistant_text": "Shanghai is located on the eastern coast of China, at the mouth of the Yangtze River. It is situated in the central part of the country, facing the East China Sea."
}


#### Dynamic system prompt

For more advanced use cases where you need to modify the system prompt based on runtime context or agent state, you can use middleware.<br>
The <i>@dynamic_prompt</i> decorator creates middleware that generates system prompts based on the model request:

In [ ]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt


agent_dyn = create_agent(
    model,
    tools = [search],
    middleware=[user_role_prompt],
    context_schema=Context
)


The system prompt will be set dynamically based on context so we get different answers to the same question.

In [119]:
query="Search for the explaination of transformers in NLP"

In [120]:
# When set user_role as "beginner"
response = agent_dyn.invoke(
    {"messages": [{"role": "user", "content":query}]},
    context={"user_role": "beginner"}
)
messages=response["messages"]
print_message_pairs(messages)

Transformers are a type of model architecture used in natural language processing (NLP) that have revolutionized the field. Here's a simple explanation:

1. **Architecture**: Transformers use a mechanism called "attention" to weigh the importance of different words in a sentence when making predictions. This allows them to understand context better than previous models.

2. **Self-Attention**: This is a key part of transformers. It means the model looks at all the words in a sentence and decides which ones are most important for understanding the meaning of each word. This helps in capturing long-range dependencies in text.

3. **Layers**: Transformers are made up of multiple layers, each consisting of attention mechanisms and feed-forward neural networks. These layers help the model learn complex patterns in data.

4. **Parallelization**: Unlike older models like RNNs (Recurrent Neural Networks), transformers can process all words in a sentence at once, rather than one at a time. This

In [121]:
# When set user_role as "expert"
response = agent_dyn.invoke(
    {"messages": [{"role": "user", "content": query}]},
    context={"user_role": "expert"}
)
messages=response["messages"]
print_message_pairs(messages)

Transformers are a type of neural network architecture that have revolutionized the field of Natural Language Processing (NLP). Introduced in the paper "Attention is All You Need" by Vaswani et al. in 2017, transformers are designed to handle sequential data and have become the foundation for many state-of-the-art NLP models, including BERT, GPT, and T5.

### Key Components of Transformers:

1. **Self-Attention Mechanism**: 
   - The self-attention mechanism allows the model to weigh the importance of different words in a sentence relative to each other. This is crucial for understanding context and relationships between words.
   - It computes attention scores for each word pair in the input sequence, enabling the model to focus on relevant parts of the input when making predictions.

2. **Positional Encoding**:
   - Since transformers do not inherently understand the order of words, positional encoding is added to input embeddings to give the model information about the position of w

#### Advanced concepts

##### ToolStrategy

ToolStrategy uses artificial tool calling to generate structured output. This works with any model that supports tool calling:

In [131]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

agent = create_agent(
    model=model,
    tools=[search],
    response_format=ToolStrategy(ContactInfo)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})


result["structured_response"]
# # ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

##### Memory

In [136]:

from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import AgentMiddleware
from typing import Any

# 假设已有 model, tool1, tool2, tools

class CustomState(AgentState):
    user_preferences: dict

class CustomMiddleware(AgentMiddleware):
    state_schema = CustomState
    tools = [search, get_weather]

    def before_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        prefs = (state.get("user_preferences") or {}) if isinstance(state, dict) else getattr(state, "user_preferences", {}) or {}
        style = prefs.get("style", "general")
        verbosity = prefs.get("verbosity", "normal")

        system_prompt = "You are a helpful assistant."
        if style == "technical":
            system_prompt += " Prefer precise, technical language and cite implementation details where relevant."
        if verbosity in ("detailed", "high"):
            system_prompt += " Provide thorough, step-by-step explanations with examples."

        return {
            "messages": [{"role": "system", "content": system_prompt}],
            "model_kwargs": {"temperature": 0.2 if style == "technical" else 0.7}
        }

agent = create_agent(
    model,
    tools=[search, get_weather],                     # 全局工具
    middleware=[CustomMiddleware()]  # 中间件（限定阶段性工具 + 注入系统提示）
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

# 读取最终回复（具体键按你的返回结构而定）
messages = result["messages"]
print(messages[-1]["content"] if isinstance(messages[-1], dict) else messages[-1].content)



Understood. I'll provide detailed technical explanations with a focus on implementation details and examples where applicable. If you have any specific topics or questions in mind, feel free to ask, and I'll ensure the response is thorough and precise.
